In [1]:
from pathlib import Path
from collections import defaultdict
import hashlib
import json
import re
import shutil
import subprocess
import zipfile
import csv

import numbers_parser
from numbers_parser import Document
from IPython.display import display


TARGET = (
    Path.home()
    / "Library/Mobile Documents/com~apple~Numbers/Documents"
    / "2026 Diary"
    / "November"
    / "November 15.numbers"
)

OUTPUT_ROOT = (
    Path.home()
    / "Downloads"
    / "Numbers Image Metadata Research"
    / "November 15"
)

EXTRACTED_DIR = OUTPUT_ROOT / "extracted_images"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)


def find_exiftool():
    candidates = [
        shutil.which("exiftool"),
        "/opt/homebrew/bin/exiftool",
        "/usr/local/bin/exiftool",
    ]

    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(Path(candidate))

    return None


EXIFTOOL = find_exiftool()


print("TARGET:")
print(TARGET)

print()
print("Exists:", TARGET.exists())

if TARGET.exists():
    print("Size:", round(TARGET.stat().st_size / 1024**2, 2), "MB")

print()
print("OUTPUT_ROOT:")
print(OUTPUT_ROOT)

print()
print("numbers-parser version:")
try:
    from importlib.metadata import version
    print(version("numbers-parser"))
except Exception:
    print("unknown")

print()
print("ExifTool:")
print(EXIFTOOL or "NOT FOUND")

TARGET:
/Users/huohsien/Library/Mobile Documents/com~apple~Numbers/Documents/2026 Diary/November/November 15.numbers

Exists: True
Size: 556.0 MB

OUTPUT_ROOT:
/Users/huohsien/Downloads/Numbers Image Metadata Research/November 15

numbers-parser version:
4.19.0

ExifTool:
NOT FOUND


In [2]:
def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def safe_filename(filename):
    filename = Path(str(filename)).name
    filename = re.sub(r"[\x00-\x1f/:]", "_", filename)
    return filename or "image"


def detect_format(data):
    h = bytes(data[:32])

    if h.startswith(b"\xff\xd8\xff"):
        return "JPEG"

    if h.startswith(b"\x89PNG\r\n\x1a\n"):
        return "PNG"

    if h.startswith((b"II*\x00", b"MM\x00*")):
        return "TIFF"

    if h.startswith((b"GIF87a", b"GIF89a")):
        return "GIF"

    if len(h) >= 12 and h[4:8] == b"ftyp":
        brand = h[8:20].lower()

        if any(
            x in brand
            for x in (
                b"heic",
                b"heix",
                b"hevc",
                b"hevx",
                b"mif1",
                b"msf1",
            )
        ):
            return "HEIC/HEIF"

        return "ISO-BMFF"

    return "unknown"


doc = Document(str(TARGET))

images = []
background_image_objects = []

image_index = 0


for sheet_index, sheet in enumerate(doc.sheets):

    print(
        f"Sheet {sheet_index}:",
        getattr(sheet, "name", None)
    )

    for table_index, table in enumerate(sheet.tables):

        print(
            f"  Table {table_index}:",
            getattr(table, "name", None)
        )

        rows = table.rows()

        for row_index, row in enumerate(rows):

            for col_index in range(len(row)):

                cell = table.cell(
                    row_index,
                    col_index
                )

                style = getattr(
                    cell,
                    "style",
                    None
                )

                if style is None:
                    continue

                bg_image = getattr(
                    style,
                    "bg_image",
                    None
                )

                if not isinstance(
                    bg_image,
                    numbers_parser.cell.BackgroundImage,
                ):
                    continue

                data = getattr(
                    bg_image,
                    "data",
                    None
                )

                if data is None:
                    continue

                data = bytes(data)

                image_index += 1

                original_filename = (
                    getattr(
                        bg_image,
                        "filename",
                        None,
                    )
                    or "image"
                )

                extracted_filename = (
                    f"{image_index:04d}"
                    f"__row{row_index + 1:04d}"
                    f"_col{col_index + 1:04d}"
                    f"__{safe_filename(original_filename)}"
                )

                extracted_path = (
                    EXTRACTED_DIR
                    / extracted_filename
                )

                # IMPORTANT:
                # exact bytes returned by numbers-parser.
                # No re-encoding.
                extracted_path.write_bytes(data)

                record = {
                    "image_index": image_index,

                    "sheet_index":
                        sheet_index,

                    "sheet_name":
                        getattr(
                            sheet,
                            "name",
                            None,
                        ),

                    "table_index":
                        table_index,

                    "table_name":
                        getattr(
                            table,
                            "name",
                            None,
                        ),

                    "numbers_row":
                        row_index + 1,

                    "numbers_col":
                        col_index + 1,

                    "original_filename":
                        original_filename,

                    "extracted_filename":
                        extracted_filename,

                    "extracted_path":
                        str(
                            extracted_path.resolve()
                        ),

                    "format":
                        detect_format(data),

                    "bytes":
                        len(data),

                    "sha256":
                        sha256_bytes(data),
                }

                images.append(record)

                #
                # Examine everything numbers-parser exposes
                # on BackgroundImage itself.
                #
                object_info = {
                    "image_index": image_index,
                    "attributes": {},
                }

                for attribute_name in dir(bg_image):

                    if attribute_name.startswith("__"):
                        continue

                    try:
                        value = getattr(
                            bg_image,
                            attribute_name,
                        )
                    except Exception as exc:

                        object_info[
                            "attributes"
                        ][attribute_name] = (
                            f"<ERROR {exc!r}>"
                        )

                        continue

                    if callable(value):
                        continue

                    if isinstance(
                        value,
                        (
                            bytes,
                            bytearray,
                            memoryview,
                        ),
                    ):

                        raw = bytes(value)

                        object_info[
                            "attributes"
                        ][attribute_name] = {
                            "type": "bytes",
                            "length": len(raw),
                            "sha256":
                                sha256_bytes(raw),
                        }

                    else:

                        object_info[
                            "attributes"
                        ][attribute_name] = (
                            repr(value)[:3000]
                        )

                background_image_objects.append(
                    object_info
                )


print()
print("TOTAL BACKGROUND IMAGES:", len(images))

print()

for record in images[:30]:
    print(
        record["image_index"],
        f'row={record["numbers_row"]}',
        f'col={record["numbers_col"]}',
        record["original_filename"],
        record["format"],
        record["bytes"],
    )


with (
    OUTPUT_ROOT
    / "background_image_objects.json"
).open(
    "w",
    encoding="utf-8",
) as fp:

    json.dump(
        background_image_objects,
        fp,
        ensure_ascii=False,
        indent=2,
    )

Sheet 0: Sheet 1
  Table 0: Table 1

TOTAL BACKGROUND IMAGES: 106

1 row=2 col=6 pasted-movie.png PNG 245196
2 row=2 col=7 pasted-movie.png PNG 100808
3 row=2 col=8 pasted-image.jpeg JPEG 3447038
4 row=2 col=9 pasted-movie.heic HEIC/HEIF 52876
5 row=5 col=3 pasted-image.jpeg JPEG 1926697
6 row=6 col=3 pasted-image.jpeg JPEG 2661957
7 row=7 col=4 pasted-image.jpeg JPEG 1355700
8 row=8 col=3 pasted-movie.heic HEIC/HEIF 98190
9 row=8 col=4 pasted-movie.heic HEIC/HEIF 153542
10 row=8 col=5 pasted-image.jpeg JPEG 2413267
11 row=9 col=3 pasted-image.jpeg JPEG 2564485
12 row=10 col=4 pasted-image.jpeg JPEG 2135317
13 row=10 col=5 pasted-image.jpeg JPEG 2289159
14 row=10 col=6 pasted-image.jpeg JPEG 3369140
15 row=10 col=7 pasted-image.tiff TIFF 28312626
16 row=10 col=8 pasted-image.tiff TIFF 28312626
17 row=11 col=5 pasted-image.jpeg JPEG 2908710
18 row=11 col=6 pasted-image.tiff TIFF 14159458
19 row=11 col=7 pasted-image.tiff TIFF 14159458
20 row=11 col=8 pasted-image.tiff TIFF 14159458
21 r

In [3]:
interesting_words = (
    "date",
    "time",
    "create",
    "capture",
    "metadata",
    "exif",
    "gps",
    "source",
    "filename",
)


for item in background_image_objects[:20]:

    interesting = {}

    for key, value in (
        item["attributes"].items()
    ):

        key_lower = key.lower()

        if any(
            word in key_lower
            for word in interesting_words
        ):
            interesting[key] = value

    print()
    print("=" * 80)
    print(
        "IMAGE",
        item["image_index"]
    )

    if interesting:

        for key, value in interesting.items():

            print()
            print(key, "=")
            print(value)

    else:

        print(
            "No date/time/metadata-like "
            "attributes exposed."
        )


IMAGE 1

_filename =
'pasted-movie.png'

filename =
'pasted-movie.png'

IMAGE 2

_filename =
'pasted-movie.png'

filename =
'pasted-movie.png'

IMAGE 3

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 4

_filename =
'pasted-movie.heic'

filename =
'pasted-movie.heic'

IMAGE 5

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 6

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 7

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 8

_filename =
'pasted-movie.heic'

filename =
'pasted-movie.heic'

IMAGE 9

_filename =
'pasted-movie.heic'

filename =
'pasted-movie.heic'

IMAGE 10

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 11

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 12

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 13

_filename =
'pasted-image.jpeg'

filename =
'pasted-image.jpeg'

IMAGE 14

_filename =
'pasted-image.j

In [5]:
# ============================================================
# Directly inspect the .numbers package.
# No Homebrew. No ExifTool.
# ============================================================

from pathlib import Path
import zipfile
import plistlib
import hashlib
import shutil


PACKAGE_OUTPUT = (
    OUTPUT_ROOT
    / "raw_numbers_package_assets"
)

PACKAGE_OUTPUT.mkdir(
    parents=True,
    exist_ok=True,
)


IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".heic",
    ".heif",
    ".png",
    ".tif",
    ".tiff",
    ".gif",
    ".webp",
    ".dng",
    ".avif",
}


def sha256_file_bytes(data):
    return hashlib.sha256(data).hexdigest()


def find_zip_member_by_suffix(
    archive,
    suffix,
):
    for info in archive.infolist():

        if info.filename.endswith(
            suffix
        ):
            return info

    return None


package_assets = []

print("TARGET:")
print(TARGET)

print()
print(
    "is file:",
    TARGET.is_file()
)

print(
    "is directory/package:",
    TARGET.is_dir()
)

print(
    "is ZIP:",
    zipfile.is_zipfile(TARGET)
)


# ------------------------------------------------------------
# Single-file .numbers = ZIP
# ------------------------------------------------------------

if zipfile.is_zipfile(TARGET):

    with zipfile.ZipFile(TARGET) as archive:

        names = archive.namelist()

        print()
        print(
            "Total package members:",
            len(names)
        )

        #
        # Show top-level package structure.
        #
        top_level = sorted({
            name.split("/")[0]
            for name in names
            if name
        })

        print()
        print("Top-level entries:")

        for name in top_level:
            print("   ", name)


        # ----------------------------------------------------
        # Numbers build/version history
        # ----------------------------------------------------

        build_info = (
            find_zip_member_by_suffix(
                archive,
                "Metadata/BuildVersionHistory.plist",
            )
        )

        if build_info:

            raw = archive.read(
                build_info
            )

            try:

                build_history = (
                    plistlib.loads(raw)
                )

                print()
                print(
                    "=" * 80
                )
                print(
                    "BuildVersionHistory.plist"
                )
                print(
                    "=" * 80
                )

                print(
                    build_history
                )

            except Exception as exc:

                print(
                    "Could not parse "
                    "BuildVersionHistory.plist:",
                    exc,
                )


        properties_info = (
            find_zip_member_by_suffix(
                archive,
                "Metadata/Properties.plist",
            )
        )

        if properties_info:

            raw = archive.read(
                properties_info
            )

            try:

                properties = (
                    plistlib.loads(raw)
                )

                print()
                print(
                    "=" * 80
                )
                print(
                    "Properties.plist"
                )
                print(
                    "=" * 80
                )

                print(
                    properties
                )

            except Exception as exc:

                print(
                    "Could not parse "
                    "Properties.plist:",
                    exc,
                )


        # ----------------------------------------------------
        # Extract Data/ image assets WITHOUT modification
        # ----------------------------------------------------

        image_index = 0

        for info in archive.infolist():

            if info.is_dir():
                continue

            member = info.filename

            #
            # Media assets normally live in Data/
            #
            if (
                not member.startswith("Data/")
                and "/Data/" not in member
            ):
                continue

            suffix = (
                Path(member)
                .suffix
                .lower()
            )

            if (
                suffix
                not in IMAGE_EXTENSIONS
            ):
                continue

            data = archive.read(info)

            image_index += 1

            output_name = (
                f"{image_index:04d}"
                f"__"
                f"{Path(member).name}"
            )

            output_path = (
                PACKAGE_OUTPUT
                / output_name
            )

            output_path.write_bytes(
                data
            )

            package_assets.append({
                "index":
                    image_index,

                "member":
                    member,

                "filename":
                    Path(member).name,

                "suffix":
                    suffix,

                "bytes":
                    len(data),

                "sha256":
                    sha256_file_bytes(data),

                "output_path":
                    str(
                        output_path.resolve()
                    ),
            })


# ------------------------------------------------------------
# Package-directory .numbers
# ------------------------------------------------------------

elif TARGET.is_dir():

    data_dirs = [
        p
        for p in TARGET.rglob("Data")
        if p.is_dir()
    ]

    print()
    print(
        "Data directories:",
        data_dirs
    )

    image_index = 0

    for data_dir in data_dirs:

        for source in (
            data_dir.rglob("*")
        ):

            if not source.is_file():
                continue

            if (
                source.suffix.lower()
                not in IMAGE_EXTENSIONS
            ):
                continue

            data = (
                source.read_bytes()
            )

            image_index += 1

            output_name = (
                f"{image_index:04d}"
                f"__"
                f"{source.name}"
            )

            output_path = (
                PACKAGE_OUTPUT
                / output_name
            )

            output_path.write_bytes(
                data
            )

            package_assets.append({
                "index":
                    image_index,

                "member":
                    str(
                        source.relative_to(
                            TARGET
                        )
                    ),

                "filename":
                    source.name,

                "suffix":
                    source.suffix.lower(),

                "bytes":
                    len(data),

                "sha256":
                    sha256_file_bytes(
                        data
                    ),

                "output_path":
                    str(
                        output_path.resolve()
                    ),
            })

else:

    raise RuntimeError(
        "Unknown Numbers package format"
    )


print()
print(
    "=" * 80
)

print(
    "RAW IMAGE ASSETS FOUND:",
    len(package_assets)
)

print(
    "Saved to:"
)

print(
    PACKAGE_OUTPUT
)

print()


for asset in package_assets[:100]:

    print(
        asset["index"],
        asset["filename"],
        asset["suffix"],
        asset["bytes"],
        asset["sha256"][:16],
    )

TARGET:
/Users/huohsien/Library/Mobile Documents/com~apple~Numbers/Documents/2026 Diary/November/November 15.numbers

is file: True
is directory/package: False
is ZIP: True

Total package members: 149

Top-level entries:
    Data
    Index
    Metadata
    preview-micro.jpg
    preview-web.jpg
    preview.jpg

BuildVersionHistory.plist
['Template: Blank (dev/15.3)', 'T15.3 (7375.0.54)', 'M14.4-7043.0.93-4', 'T15.3 (7375.0.54)', 'M14.4-7043.0.93-4', 'T15.3 (7375.0.54)', 'M14.4-7043.0.93-4', 'T15.3 (7375.0.54)', 'M14.4-7043.0.93-4', 'T15.3 (7375.0.54)', 'M14.4-7043.0.93-4', 'T15.3 (7375.0.54)', 'M14.4-7043.0.93-4', 'T15.3 (7375.0.54)']

Properties.plist
{'revision': '0::75DA23FD-6354-4727-A260-C5A2E94471D5', 'hasExternalReferenceOrMissingData': False, 'documentUUID': '6CB25092-45CE-4421-AA7B-1048256A0FD2', 'hasUnmaterializedRemoteData': False, 'versionUUID': '75DA23FD-6354-4727-A260-C5A2E94471D5', 'privateUUID': 'AB50C69A-DA5F-429E-B0EE-075A3900B9DB', 'isMultiPage': False, 'stableDocumen

In [9]:
# ============================================================
# Inspect embedded metadata DIRECTLY INSIDE November 15.numbers
# No extraction to disk.
# No Homebrew.
# No ExifTool.
# ============================================================

from io import BytesIO
from pathlib import Path
import zipfile

from PIL import Image, ExifTags

import pillow_heif
pillow_heif.register_heif_opener()

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".tif",
    ".tiff",
    ".png",
    ".heic",
    ".heif",
}


# EXIF numeric tag IDs
DATE_TAG_IDS = {
    306:   "DateTime",
    36867: "DateTimeOriginal",
    36868: "DateTimeDigitized",
    36880: "OffsetTime",
    36881: "OffsetTimeOriginal",
    36882: "OffsetTimeDigitized",
}

CAMERA_TAG_IDS = {
    271: "Make",
    272: "Model",
    42036: "LensModel",
}


def readable_exif_dict(exif):
    result = {}

    for tag_id, value in exif.items():

        tag_name = ExifTags.TAGS.get(
            tag_id,
            str(tag_id),
        )

        result[tag_name] = value

    return result


results = []


with zipfile.ZipFile(TARGET) as archive:

    image_members = [
        info
        for info in archive.infolist()
        if (
            not info.is_dir()
            and info.filename.startswith("Data/")
            and Path(info.filename).suffix.lower()
                in IMAGE_EXTENSIONS
        )
    ]

    print(
        "Image members:",
        len(image_members)
    )

    for index, info in enumerate(
        image_members,
        start=1,
    ):

        #
        # IMPORTANT:
        # bytes remain INSIDE RAM.
        # Nothing is written to disk.
        #
        data = archive.read(info)

        row = {
            "index": index,
            "member": info.filename,

            # ZIP member timestamp.
            # This is NOT assumed to be capture time.
            "zip_member_time":
                info.date_time,

            "bytes":
                len(data),

            "format":
                None,

            "DateTime":
                None,

            "DateTimeOriginal":
                None,

            "DateTimeDigitized":
                None,

            "OffsetTime":
                None,

            "OffsetTimeOriginal":
                None,

            "OffsetTimeDigitized":
                None,

            "Make":
                None,

            "Model":
                None,

            "LensModel":
                None,

            "error":
                None,
        }


        try:

            with Image.open(
                BytesIO(data)
            ) as image:

                row["format"] = (
                    image.format
                )

                exif = (
                    image.getexif()
                )

                #
                # First inspect the normal EXIF mapping.
                #
                for tag_id, name in {
                    **DATE_TAG_IDS,
                    **CAMERA_TAG_IDS,
                }.items():

                    if tag_id in exif:

                        row[name] = (
                            exif.get(tag_id)
                        )


                #
                # DateTimeOriginal usually lives
                # inside the nested Exif IFD.
                #
                try:

                    exif_ifd = (
                        exif.get_ifd(
                            34665
                        )
                    )

                except Exception:

                    exif_ifd = {}


                for tag_id, name in (
                    DATE_TAG_IDS.items()
                ):

                    if (
                        row.get(name) is None
                        and
                        tag_id in exif_ifd
                    ):

                        row[name] = (
                            exif_ifd[tag_id]
                        )


                for tag_id, name in (
                    CAMERA_TAG_IDS.items()
                ):

                    if (
                        row.get(name) is None
                        and
                        tag_id in exif_ifd
                    ):

                        row[name] = (
                            exif_ifd[tag_id]
                        )


        except Exception as exc:

            row["error"] = (
                repr(exc)
            )


        results.append(row)


# ------------------------------------------------------------
# Print only interesting results
# ------------------------------------------------------------

for row in results:

    print()
    print("=" * 100)

    print(
        row["index"],
        row["member"]
    )

    print(
        "ZIP member timestamp :",
        row["zip_member_time"]
    )

    print(
        "Pillow format        :",
        row["format"]
    )

    print(
        "DateTimeOriginal     :",
        row["DateTimeOriginal"]
    )

    print(
        "DateTimeDigitized    :",
        row["DateTimeDigitized"]
    )

    print(
        "DateTime             :",
        row["DateTime"]
    )

    print(
        "OffsetTimeOriginal   :",
        row["OffsetTimeOriginal"]
    )

    print(
        "Camera               :",
        row["Make"],
        row["Model"]
    )

    print(
        "Lens                 :",
        row["LensModel"]
    )

    if row["error"]:

        print(
            "ERROR                :",
            row["error"]
        )


print()
print("=" * 100)
print("SUMMARY")
print("=" * 100)

print(
    "Total:",
    len(results)
)

print(
    "DateTimeOriginal:",
    sum(
        row["DateTimeOriginal"]
        is not None
        for row in results
    )
)

print(
    "Any EXIF date:",
    sum(
        any(
            row.get(name) is not None
            for name in (
                "DateTimeOriginal",
                "DateTimeDigitized",
                "DateTime",
            )
        )
        for row in results
    )
)

print(
    "Camera Make/Model:",
    sum(
        row["Make"] is not None
        or row["Model"] is not None
        for row in results
    )
)

print(
    "Pillow couldn't decode:",
    sum(
        row["error"] is not None
        for row in results
    )
)

Image members: 121

1 Data/pasted-movie-16.png
ZIP member timestamp : (2026, 11, 15, 12, 26, 40)
Pillow format        : PNG
DateTimeOriginal     : 2026:11:15 07:16:20
DateTimeDigitized    : None
DateTime             : None
OffsetTimeOriginal   : None
Camera               : None None
Lens                 : None

2 Data/pasted-movie-18.png
ZIP member timestamp : (2026, 11, 15, 12, 26, 56)
Pillow format        : PNG
DateTimeOriginal     : 2026:11:15 07:21:58
DateTimeDigitized    : None
DateTime             : None
OffsetTimeOriginal   : None
Camera               : None None
Lens                 : None

3 Data/pasted-image-20.jpeg
ZIP member timestamp : (2026, 11, 15, 12, 27, 6)
Pillow format        : MPO
DateTimeOriginal     : 2026:11:15 09:49:03
DateTimeDigitized    : 2026:11:15 09:49:03
DateTime             : 2026:11:15 09:49:03
OffsetTimeOriginal   : +08:00
Camera               : Apple iPhone 18 Pro Max
Lens                 : iPhone 18 Pro Max back triple camera 6.93mm f/1.8

4 Data/pas

In [11]:
# ============================================================
# Compare numbers-parser BackgroundImage.data
# against raw Data/* assets inside the .numbers ZIP.
# ============================================================

import hashlib
import zipfile
from collections import defaultdict
from pathlib import Path

import numbers_parser
from numbers_parser import Document


def sha256_bytes(data):
    return hashlib.sha256(
        bytes(data)
    ).hexdigest()


# ------------------------------------------------------------
# 1. Hash every raw image asset directly inside .numbers/Data
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".heic",
    ".heif",
    ".tif",
    ".tiff",
    ".gif",
    ".webp",
    ".dng",
    ".avif",
}


raw_assets_by_sha = defaultdict(list)

with zipfile.ZipFile(TARGET) as archive:

    for info in archive.infolist():

        if info.is_dir():
            continue

        if not info.filename.startswith(
            "Data/"
        ):
            continue

        if (
            Path(info.filename)
            .suffix
            .lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        data = archive.read(info)

        digest = sha256_bytes(data)

        raw_assets_by_sha[digest].append({
            "member":
                info.filename,

            "bytes":
                len(data),

            "zip_timestamp":
                info.date_time,
        })


# ------------------------------------------------------------
# 2. Hash every numbers-parser BackgroundImage.data
# ------------------------------------------------------------

doc = Document(str(TARGET))

parser_images = []

image_index = 0


for sheet in doc.sheets:

    for table in sheet.tables:

        rows = table.rows()

        for row_index, row in enumerate(
            rows
        ):

            for col_index in range(
                len(row)
            ):

                cell = table.cell(
                    row_index,
                    col_index,
                )

                style = getattr(
                    cell,
                    "style",
                    None,
                )

                bg = (
                    getattr(
                        style,
                        "bg_image",
                        None,
                    )
                    if style is not None
                    else None
                )

                if not isinstance(
                    bg,
                    numbers_parser.cell.BackgroundImage,
                ):
                    continue

                data = getattr(
                    bg,
                    "data",
                    None,
                )

                if data is None:
                    continue

                data = bytes(data)

                image_index += 1

                digest = sha256_bytes(
                    data
                )

                matches = (
                    raw_assets_by_sha.get(
                        digest,
                        [],
                    )
                )

                parser_images.append({
                    "index":
                        image_index,

                    "row":
                        row_index + 1,

                    "col":
                        col_index + 1,

                    "bg_filename":
                        getattr(
                            bg,
                            "filename",
                            None,
                        ),

                    "bytes":
                        len(data),

                    "sha256":
                        digest,

                    "exact_match":
                        bool(matches),

                    "raw_matches":
                        matches,
                })


# ------------------------------------------------------------
# 3. Report
# ------------------------------------------------------------

matched = sum(
    item["exact_match"]
    for item in parser_images
)

print(
    "numbers-parser images:",
    len(parser_images),
)

print(
    "raw Data/ image SHA values:",
    len(raw_assets_by_sha),
)

print(
    "EXACT SHA-256 matches:",
    matched,
    "/",
    len(parser_images),
)

print()


for item in parser_images:

    if not item["exact_match"]:

        print(
            "NO MATCH:",
            item["index"],
            "row",
            item["row"],
            "col",
            item["col"],
            item["bg_filename"],
            item["bytes"],
            item["sha256"],
        )

numbers-parser images: 121
raw Data/ image SHA values: 121
EXACT SHA-256 matches: 121 / 121

